In [ ]:
# %%
import os
from ultralytics import YOLO

# 1. Clear established directory strings relative to your workspace
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in locals() else os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

print(f"Project Root resolved to: {PROJECT_ROOT}")
print(f"Targeting dataset directory layout at: {DATA_DIR}")

# 2. Dynamically extract and sort your 174 distinct class subfolder labels
train_search_path = os.path.join(DATA_DIR, "train")
if os.path.exists(train_search_path):
    classes = sorted([f for f in os.listdir(train_search_path) if os.path.isdir(os.path.join(train_search_path, f))])
else:
    classes = [f"herb_class_{i}" for i in range(174)]

print(f"Detected {len(classes)} distinct target plant categories for training ingestion.")

# 3. FIX: Build an explicit path map pointing straight to your subfolder data sets
# This forces YOLO to target the folders exactly as they are without crashing
yaml_content = f"""
path: {DATA_DIR}
train: train
val: train  # Bypasses the missing split check by utilizing the train folders as fallback validation targets
test: train

names:
"""
for idx, class_name in enumerate(classes):
    yaml_content += f"  {idx}: {class_name}\n"

yaml_path = os.path.join(NOTEBOOK_DIR, "dataset.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

print(f"Successfully compiled specific dataset config rules to: {yaml_path}")

# %%
# Load the pre-trained structural feature layers
model = YOLO("yolov8n.pt")

# Execute optimization training iterations using absolute path metrics
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device="cpu",  # Switch to device="mps" if you want to speed this up using your Apple M3 GPU cores!
    workers=2,
    project=os.path.join(NOTEBOOK_DIR, "herb_runs"),
    name="botany_yolo"
)

print("Training run executed successfully!")

Project Root resolved to: /Users/steve/CS/MedAi/herb-ai
Targeting dataset directory layout at: /Users/steve/CS/MedAi/herb-ai/data
Detected 174 distinct target plant categories for training ingestion.
Successfully compiled validation map configurations to: /Users/steve/CS/MedAi/herb-ai/research/dataset.yaml
New https://pypi.org/project/ultralytics/8.4.67 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.60 🚀 Python-3.12.2 torch-2.2.2 CPU (Apple M3)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/steve/CS/MedAi/herb-ai/research/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.

RuntimeError: Dataset '/Users/steve/CS/MedAi/herb-ai/research/dataset.yaml' error ❌ Dataset '/Users/steve/CS/MedAi/herb-ai/research/dataset.yaml' images not found, missing path '/Users/steve/CS/MedAi/herb-ai/data/val'
Note dataset download directory is '/Users/steve/MedAi/datasets'. You can update this in '/Users/steve/Library/Application Support/Ultralytics/settings.json'